# Generalized Saito Criterion and Determinant Ideals

Notation: for a derivation matrix $M$ and defining polynomial $Q(\mathcal{A})$, write $\Delta_I = g_I Q(\mathcal{A})$.  The notebook distinguishes the unscaled determinant ideal $\langle \Delta_I\rangle$ from the scaled minor ideal $\langle g_I\rangle$.


| Section (Link) | What it is meant to show |
|---|---|
| [Example 1](#Example-1:-Free-Arrangements-(Classical-Saito-Criterion)) | Classical Saito criterion for free arrangements: $\det M / Q$ is a nonzero constant. |
| [Example 2](#Example-2:-A-3-Dimensional-SPOG-Arrangement) | A minimal 3-dimensional SPOG arrangement: the scaled cofactors $g_i$ give the unique relation and satisfy the SPOG criterion. |
| [Example 3](#Example-3:-3-Dimensional) | Deletions in dimension 3: the criterion detects which deletions are SPOG and which are not. |
| [Example 4](#Example-4:-4-Dimensional-SPOG-Arrangement) | A 4-dimensional SPOG example: in higher dimension the free resolution / projective dimension must be checked explicitly. |
| [Example 5](#Example-5:-Generic-Arrangements-and-the-Determinant-Ideal-Conjecture) | Generic arrangements: the conjecture is that $\langle\Delta_I\rangle = Q\cdot S_d$, equivalently $\langle g_I\rangle = S_d$ for $d=(\ell-2)k$. |
| [Example 6](#Example-6:-NTF-Deletions-and-Maximal-Minors) | An NTF deletion example: maximal minors and the criterion in a nontrivial 4-dimensional deletion family. |
| [Example 7](#Example-7:-Two-Hyperplane-Deletion-in-Dimension-4) | A two-hyperplane deletion: cofactor relations can be linear while selected scaled determinants are quadratic. |
| [Example 8](#Example-8:-Stress-Tests) | Stress tests: arbitrary generating candidates pass the height test, while low-height candidates do not. |
| [Example 9](#Example-9:-$\ell+1$-generators-but-Not-Strictly-Plus-One) | Plus-one-generated but not strictly plus-one: generation can hold even when no $g_i$ is linear. |
| [Example 10](#Example-10:-$\operatorname{pd}D(\mathcal{A})=1$-but-rank-2-syzygies) | $\operatorname{pd}_S D(\mathcal{A})=1$ with $\ell+2$ minimal generators (rank-$2$ syzygies): no $(\ell+1)$-subset satisfies the height hypothesis. |
| [Example 11](#Example-11:-Conjectural-Height-Bound-$\operatorname{ht}-I_\mathcal{A}(G)\ge\operatorname{pd}_S-D(\mathcal{A})+2$) | Conjectural height bound $\operatorname{ht}\,I_\mathcal{A}(G)\ge \operatorname{pd}_S D(\mathcal{A})+2$: tested across $\operatorname{pd}=0,1$ (baselines) and several $\operatorname{pd}=2, 3$ cases. |

> **Companion notebook** [`experiment_saito.ipynb`](experiment_saito.ipynb): further experiments — the full $(\ell+2)$-tuple investigation extending Example 10, and the unified Plücker-tensor formulation of the Saito-type criterion.


In [14]:
from IPython.display import Math, display
import sage.misc.latex as sage_latex
import itertools
import sys
from sage.modules.free_module_element import FreeModuleElement_generic_dense as module_elem
from sage.libs.singular.function_factory import singular_function
syz = singular_function("syz")

from pathlib import Path

def _find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in (start, *start.parents):
        if (path / 'src' / 'hyperplane_arrangements').is_dir():
            return path
    raise RuntimeError('Run this notebook from inside the repository or install the package with sage -pip install -e .')

ROOT = _find_repo_root()
src = ROOT / 'src'
if str(src) not in sys.path:
    sys.path.insert(0, str(src))
from hyperplane_arrangements import *

try:
    original_show = show
except NameError:
    raise SystemExit("Switch to the SageMath kernel and re-run.")

def vscode_show(*args, **kwargs):
    for expr in args:
        try:
            if hasattr(expr, '_latex_'):
                display(Math(expr._latex_()))
            else:
                display(Math(sage_latex.latex(expr)))
        except Exception as e:
            print(f"LaTeX rendering failed: {e}")
            original_show(expr, **kwargs)
show = vscode_show
print(sage.version.banner)

def _generator_list(A, generators=None):
    return list(A.minimal_generators() if generators is None else generators)


def report_scaled_minor_checks(A, generators=None, label="G", verify=False):
    """Print the full scaled-minor height and check all ell+1 sublists."""
    G = _generator_list(A, generators)
    ell = A.n
    p = len(G)
    I = A.scaled_minor_ideal(G)
    ht = A.scaled_minor_ideal_height(G)
    print(f"{label}: p={p}, ell={ell}, ht I_A({label})={ht}, proper={not I.is_one()}")

    if p < ell + 1:
        print("  no ell+1 sublists to test")
        return []

    subset_results = []
    for J in itertools.combinations(range(p), ell + 1):
        sub = [G[j] for j in J]
        res = A.check_generalized_saito(generators=sub, verify=verify, verbose=False)
        subset_results.append((J, res))

    height_counts = {}
    for _, res in subset_results:
        height_counts[res['height']] = height_counts.get(res['height'], 0) + 1
    n_criterion = sum(1 for _, res in subset_results if res['criterion_applies'])
    n_spog = sum(1 for _, res in subset_results if res['predicts_minimal_spog'])
    print(f"  checked C({p},{ell + 1}) = {len(subset_results)} sublists")
    print(f"  height distribution: {height_counts}")
    print(f"  criterion applies: {n_criterion}; predicts minimal SPOG: {n_spog}")
    if verify:
        n_generate = sum(1 for _, res in subset_results if res.get('actually_generates'))
        print(f"  actually generates: {n_generate}")
    if len(subset_results) <= 20:
        for J, res in subset_results:
            print(f"    {J}: ht={res['height']}, criterion={res['criterion_applies']}, predicts_min_SPOG={res['predicts_minimal_spog']}")
    return subset_results


SageMath version 10.7, Release Date: 2025-08-09


## Example 1: Free Arrangements (Classical Saito Criterion)

**What this shows.** In the free case the derivation module has exactly $\ell$ generators.  Saito's classical criterion says that the determinant of the generator matrix is a unit multiple of $Q(\mathcal{A})$.


In [2]:
# Boolean arrangement in dimension 3: free with exponents (1,1,1)
A_free = HyperplaneArrangement([[1,0,0],[0,1,0],[0,0,1]])
print(f"Free: {A_free.is_free}, degree seq: {A_free.degrees()}")
m = matrix([g.v for g in A_free.minimal_generators()])
show(m)
print(f"det(M) / Q = {m.det() // A_free.Q}")

Free: True, degree seq: [1, 1, 1]


<IPython.core.display.Math object>

det(M) / Q = -3


In [3]:
# A-2 arrangement: free with exponents (1,2,3)
A2 = HyperplaneArrangement([[1,0,0],[0,1,0],[0,0,1],[1,-1,0],[1,0,-1],[0,1,-1]])
print(f"Free: {A2.is_free}, degree seq: {A2.degrees()}")
m2 = matrix([g.v for g in A2.minimal_generators()])
show(m2)
det_coeff = m2.det() // A2.Q
print(f"det(M) / Q = {det_coeff} (nonzero constant => free by Saito)")

Free: True, degree seq: [1, 2, 3]


<IPython.core.display.Math object>

det(M) / Q = 12 (nonzero constant => free by Saito)


## Example 2: A 3-Dimensional SPOG Arrangement

**What this shows.** For $\ell=3$, the plus-one case can be checked directly from the scaled cofactors $g_i$.  The same $g_i$ also give the unique relation $\sum_i g_i\theta_i=0$ among the four minimal generators.


In [4]:
from IPython.display import display, Markdown
A = HyperplaneArrangement([[1,0,0],[0,1,0],[0,0,1],[1,1,1],[1,1,2]])
print(f"Num planes: {A.num_planes}, Degree seq: {A.degrees()}")
print(f"Free: {A.is_free}, SPOG: {A.is_spog()}")
display(Markdown('**Output of `show(A.free_resolution())`:**'))
display(show(A.free_resolution()))

Num planes: 5, Degree seq: [1, 2, 3, 3]
Free: False, SPOG: [1, 2, 3, 3]


<IPython.core.display.Math object>

In [5]:
# Compute all Delta_I and Saito coefficients
deltas = A.delta_I()
print("All maximal minors Delta_I = g_I * Q:")
for I, (delta, g) in sorted(deltas.items()):
    print(f"  I = {I}: g_I = {g}")


All maximal minors Delta_I = g_I * Q:
  I = (0, 1, 2): g_I = 25*x1
  I = (0, 1, 3): g_I = -25*x0 - 50*x1
  I = (0, 2, 3): g_I = -75*x1^2 + 150*x1*x2 - 50*x2^2
  I = (1, 2, 3): g_I = 0


In [6]:
# Check the generalized Saito criterion.
# For p = ell+1, saito_coefficients() returns the signed cofactors g_i.
coeffs = A.saito_coefficients()
print("Saito coefficients g_i:")
for i, g in enumerate(coeffs):
    print(f"  g_{i} = {g} (degree {g.degree() if g != 0 else -1})")

print()
result = A.check_generalized_saito(verbose=True)


Saito coefficients g_i:
  g_0 = 0 (degree -1)
  g_1 = 75*x1^2 - 150*x1*x2 + 50*x2^2 (degree 2)
  g_2 = 25*x0 + 50*x1 (degree 1)
  g_3 = -25*x1 (degree 1)

  scaled minors (g_i):
    g_0 = 0  (deg -1)
    g_1 = 75*x1^2 - 150*x1*x2 + 50*x2^2  (deg 2)
    g_2 = 25*x0 + 50*x1  (deg 1)
    g_3 = -25*x1  (deg 1)
  proper:                     True
  ht I_A(G):                  3
  criterion applies (ht>=3):  True
  all g_i of positive degree: True
  some g_i is linear:         True
  predicts minimal SPOG:      True
g_{ell+1} = -25*x1
  g_0 = 0 (degree -1)
  g_1 = 75*x1^2 - 150*x1*x2 + 50*x2^2 (degree 2)
  g_2 = 25*x0 + 50*x1 (degree 1)
  gcd(g_0,...,g_{ell-1}) mod g_{ell} = 1
  actually generates D(A):    True


In [7]:
from IPython.display import display, Markdown
# Verify the relation sum g_i theta_i = 0
m = matrix([g.v for g in A.minimal_generators()])
relation = sum(coeffs[i] * m[i] for i in range(len(coeffs)))
print(f"Relation sum g_i * theta_i = {relation}")

# Compare with syz computation
print("\nRelations from syz:")
display(Markdown('**Output of `show(matrix(A.relations()))`:**'))
display(show(matrix(A.relations())))

Relation sum g_i * theta_i = (0, 0, 0)

Relations from syz:


<IPython.core.display.Math object>

## Example 3: 3-Dimensional

**What this shows.** Starting from a non-free arrangement, different single deletions can be free, SPOG, or neither.  The criterion is used here as a classifier for the SPOG deletions.


In [8]:
# Start from a larger arrangement and take deletions.
mat = [[1,0,0],[0,1,0],[0,0,1],[1,1,1],[1,1,2],[1,2,1],[2,1,1]]
A_big = HyperplaneArrangement(mat)
print(f"Full arrangement: {A_big.num_planes} planes, degrees = {A_big.degrees()}, free = {A_big.is_free}, SPOG = {A_big.is_spog()}")

for i in range(A_big.num_planes):
    B = A_big.deletion([i])
    G = list(B.minimal_generators())
    spog = B.is_spog()
    if spog:
        print(f"\nDeletion of plane {i}: SPOG with degrees {spog}")
        res = B.check_generalized_saito(verbose=False)
        print(f"  Criterion predicts minimal SPOG: {res['predicts_minimal_spog']}")
    elif B.is_free:
        print(f"\nDeletion of plane {i}: Free with degrees {B.degrees()}")
    else:
        print(f"\nDeletion of plane {i}: neither free nor SPOG, degrees = {B.degrees()}")

    if len(G) > B.n + 1:
        report_scaled_minor_checks(B, G, label=f"delete {i}", verify=False)


Full arrangement: 7 planes, degs = [1, 4, 4, 4], free = False, SPOG = False

Deletion of plane 0: SPOG with degs [1, 3, 3, 4]
  Criterion predicts minimal SPOG: True

Deletion of plane 1: SPOG with degs [1, 3, 3, 4]
  Criterion predicts minimal SPOG: True

Deletion of plane 2: SPOG with degs [1, 3, 3, 4]
  Criterion predicts minimal SPOG: True

Deletion of plane 3: neither free nor SPOG, degs = [1, 4, 4, 4, 4, 4]
delete 3: p=6, ell=3, ht I_A(delete 3)=3, proper=True
  checked C(6,4) = 15 sublists
  height distribution: {1: 1, 2: 9, 0: 5}
  criterion applies: 0; predicts minimal SPOG: 0
    (0, 1, 2, 3): ht=1, criterion=False, predicts_min_SPOG=False
    (0, 1, 2, 4): ht=2, criterion=False, predicts_min_SPOG=False
    (0, 1, 2, 5): ht=2, criterion=False, predicts_min_SPOG=False
    (0, 1, 3, 4): ht=2, criterion=False, predicts_min_SPOG=False
    (0, 1, 3, 5): ht=2, criterion=False, predicts_min_SPOG=False
    (0, 1, 4, 5): ht=2, criterion=False, predicts_min_SPOG=False
    (0, 2, 3, 4):

## Example 4: 4-Dimensional SPOG Arrangement


In [12]:
from IPython.display import display, Markdown
# 4-dimensional arrangement
mat4 = coordinate_vectors(4) + [[1,1,1,0]]
A4 = HyperplaneArrangement(mat4)
print(f"Num planes: {A4.num_planes}, Degree seq: {A4.degrees()}")
print(f"Free: {A4.is_free}, SPOG: {A4.is_spog()}")
display(Markdown('**Output of `show(A4.free_resolution())`:**'))
display(show(A4.free_resolution()))

Num planes: 5, Degree seq: [1, 1, 2, 2, 2]
Free: False, SPOG: [1, 1, 2, 2, 2]


<IPython.core.display.Math object>

In [ ]:
if A4.is_spog():
    print("Checking generalized Saito criterion:")
    res4 = A4.check_generalized_saito(verbose=True)
else:
    print("Arrangement is not SPOG; trying deletions...")
    for i in range(A4.num_planes):
        B = A4.deletion([i])
        if B.is_spog():
            print(f"\nDeletion of plane {i}: SPOG with degrees {B.is_spog()}")
            res = B.check_generalized_saito(verbose=True)
            break


## Example 5: Generic Arrangements and the Determinant-Ideal Conjecture

**Conjecture (generic case).** 
Let $\mathcal{A}$ be a central arrangement in $\mathbb{K}^\ell$ with $|\mathcal{A}|=\ell+k$, and set $d=(\ell-2)k$.  

$\mathcal{A}$ is generic if and only if

for the maximal minors of the minimal-generator matrix,

$$\langle \Delta_I\rangle = Q(\mathcal{A})\cdot S_d.$$

Equivalently, after dividing by $Q(\mathcal{A})$, the scaled minor ideal satisfies

$$\langle g_I\rangle = S_d,$$

where $S_d$ denotes the vector space of homogeneous degree-$d$ polynomials.  


In computations below this is tested by checking that every monomial of degree $d$ lies in $\langle g_I\rangle$; the expected number of such monomials is $\binom{\ell+d-1}{\ell-1}$.


In [ ]:
# Test the generic determinant-ideal conjecture in dimensions 3 and 4.
# scaled_minor_ideal() returns <g_I>, where g_I = det(M_I) / Q.
for n in [3, 4]:
    print(f"=== Dimension {n} ===")
    for k in range(1, 4):
        mat_gen = create_generic_arrangement(n, n + k)
        A_gen = HyperplaneArrangement(mat_gen)
        G = list(A_gen.minimal_generators())
        d = (n - 2) * k
        expected_dim = binomial(n + d - 1, n - 1)
        print(f"\n  |A|={n+k}, k={k}, d={d}")
        print(f"  degrees={A_gen.degrees()}, free={A_gen.is_free}, SPOG={bool(A_gen.is_spog())}")

        scaled_minor_ideal = A_gen.scaled_minor_ideal(G)
        GB = scaled_minor_ideal.groebner_basis()
        degree_d_monomials = [exponent_to_polynomial(u, A_gen.v) for u in sk_expo(d, n)]
        missing = [mon for mon in degree_d_monomials if mon not in scaled_minor_ideal]

        print(f"  conjecture: <Delta_I> = Q*S_{d}, equivalently <g_I> = S_{d}")
        print(f"  ht <g_I> = {A_gen.scaled_minor_ideal_height(G)}")
        print(f"  |GB(<g_I>)|={len(GB)}, dim S_{d}={expected_dim}")
        print(f"  all degree-{d} monomials in <g_I>: {len(missing) == 0}")
        if missing:
            print(f"  first missing monomial: {missing[0]}")
        if len(G) > A_gen.n + 1:
            report_scaled_minor_checks(A_gen, G, label=f"generic n={n}, k={k}", verify=False)
    print()


## Example 6: NTF Deletions and Maximal Minors

**What this shows.** This keeps the nontrivial NTF deletion experiment from the old notebook, but avoids listing every duplicated determinant computation.  We scan deletions and then inspect one SPOG deletion with the criterion.


In [ ]:
# NTF deletion survey, summarized to one line per deletion.
mat_ntf = coordinate_vectors(4) + [[1,-1,0,0],[1,0,-1,0],[1,0,0,-1],[0,1,-1,0],[0,1,0,-1],[0,0,1,-1],[0,1,-1,1],[1,-1,1,-1]]
A_ntf = HyperplaneArrangement(mat_ntf)
print(f"Full NTF arrangement: {A_ntf.num_planes} planes, degrees = {A_ntf.degrees()}, free = {A_ntf.is_free}")

first_spog = None
for i in range(A_ntf.num_planes):
    B = A_ntf.deletion([i])
    G = list(B.minimal_generators())
    spog = B.is_spog()
    label = "SPOG" if spog else ("free" if B.is_free else "neither")
    print(f"  delete {i}: {label}, degrees = {B.degrees()}")
    if len(G) > B.n + 1:
        report_scaled_minor_checks(B, G, label=f"NTF delete {i}", verify=False)
    if spog and first_spog is None:
        first_spog = (i, B)

if first_spog is not None:
    i, B = first_spog
    print(f"\nInspecting the first SPOG deletion: delete {i}")
    print(f"free resolution: {B.free_resolution()}")
    res = B.check_generalized_saito(verbose=True)


## Example 7: Two-Hyperplane Deletion in Dimension 4

**What this shows.** A two-hyperplane deletion can have a linear relation among generators while selected scaled maximal determinants have degree 2.  This separates the syzygy degree from the degrees of individual cofactors.


In [ ]:
mat10 = [
    [1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0],
    [-1, 1, 0, 0], [-1, 0, 1, 0], [0, -1, 1, 0],
    [0, 0, 0, 1], [0, 1, -1, 1], [-1, 1, -1, 1], [0, 0, -1, 1]
]
A10 = HyperplaneArrangement(mat10)
print(f"Full arrangement: {A10.num_planes} planes, degrees = {A10.degrees()}, free = {A10.is_free}")

# The old experiment tested J=[3,5] and noted J=[2,9] as a pd=2 contrast.
J = [3, 5]
B10 = A10.deletion(J)
print(f"Deletion of {J}: {B10.num_planes} planes, degrees = {B10.degrees()}")
print(f"Free: {B10.is_free}, SPOG: {B10.is_spog()}")
show(B10.free_resolution())


In [17]:
# Syzygies among the generators
m = matrix(B10.minimal_generators())
relations = syz(Sequence([module_elem(B10.S**B10.n, tuple(m[i])) for i in range(m.nrows())]))
print(f"Number of generators: {m.nrows()}, Number of relations: {len(relations)}")
for i, r in enumerate(relations):
    print(f"  relation {i}: {r}")

# Verify the cofactor relation: sum (-1)^i det(M_{[5]\{i}}) / Q * theta_i = 0
# relations are deg-1 but all determinants are deg-2
print("\nCofactor relation check:")
II = list(range(min(5, m.nrows())))
a = 0
for i in range(len(II)):
    I = II.copy()
    del I[i]
    b = (-1)**i * det(m[I]) / B10.Q
    a += b * m[II[i]]
    show(factor(b) if b else 0)
print(f"sum g_i * theta_i = {a}")

Number of generators: 7, Number of relations: 4
  relation 0: (0, 8*x1 - 8*x3, -16*x2 + 8*x3, 0, -x3, 5*x3, 0)
  relation 1: (0, 24*x0 - 108*x3, 72*x3, -24*x2 + 24*x3, 5*x3, -9*x3, 4*x3)
  relation 2: (0, 0, 16*x0 - 72*x3, -8*x1 + 8*x3, 3*x3, x3, 0)
  relation 3: (0, 0, -72*x1 + 72*x3, -12*x1 + 12*x3, -3*x0 - 5*x1 + 9*x2 + 14*x3, 15*x0 + 9*x1 + 3*x2 - 78*x3, -4*x1 + 4*x3)

Cofactor relation check:


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

sum g_i * theta_i = (0, 0, 0, 0)


In [ ]:
# Scaled minor ideal and specific maximal minors.
print("Scaled minor ideal GB:")
show(B10.scaled_minor_ideal().groebner_basis())
if len(B10.minimal_generators()) > B10.n + 1:
    report_scaled_minor_checks(B10, label="B10", verify=False)

# Selected maximal minors.
print("\nSample scaled maximal minors det(M_I) / Q:")
for J_sub in [[2,3,4,5], [0,1,2,5]]:
    val = m[J_sub, :].det() // B10.Q
    print(f"  I = {J_sub}:", end=" ")
    show(val)


## Example 8: Stress Tests

**What this shows.** The generalized criterion applies to arbitrary candidate $(\ell+1)$-tuples in $D(\mathcal{A})$, not only to the canonical minimal generators.  We perturb a known generating set by elementary $S$-row operations and verify that the height criterion still matches actual generation.


In [20]:
import random
random.seed(int(0))

A = HyperplaneArrangement([[1,0,0],[0,1,0],[0,0,1],[1,1,1],[1,1,2]])
G = list(A.minimal_generators().gens)
x = A.v

# Build perturbed candidates: replace G[i] by G[i] + h*G[j] (i != j)
trials = []
for i, j, h in [(1, 0, x[0]), (2, 0, x[1]**2), (3, 0, x[2]**2),
                (1, 3, x[2]), (2, 3, x[0]+x[1])]:
    Gp = list(G)
    Gp[i] = G[i] + h*G[j]
    res = A.check_generalized_saito(generators=Gp, verbose=False)
    trials.append((i, j, h, res['criterion_applies'], res['actually_generates']))
    print(f"  swap G[{i}] += ({h})*G[{j}] : "
          f"crit={res['criterion_applies']}, "
          f"generates={res['actually_generates']}, "
          f"ht={res['height']}, predicts_min_SPOG={res['predicts_minimal_spog']}")
assert all(c == g for _, _, _, c, g in trials), "Criterion disagrees with actual generation!"
print("\nAll perturbations: criterion matches actual generation.")

  swap G[1] += (x0)*G[0] : crit=True, generates=True, ht=3, predicts_min_SPOG=True


  swap G[2] += (x1^2)*G[0] : crit=True, generates=True, ht=3, predicts_min_SPOG=True
  swap G[3] += (x2^2)*G[0] : crit=True, generates=True, ht=3, predicts_min_SPOG=True
  swap G[1] += (x2)*G[3] : crit=True, generates=True, ht=3, predicts_min_SPOG=True
  swap G[2] += (x0 + x1)*G[3] : crit=True, generates=True, ht=3, predicts_min_SPOG=True

All perturbations: criterion matches actual generation.


### Searching for Counterexamples

The implication tested here is `criterion_applies => G generates D(A)`.  The examples use elementary perturbations that preserve generation and degenerate candidates that should make the height hypothesis fail.


In [22]:
import random
random.seed(int(42))

def elementary_perturbations(A, base_gens, n_trials=15, deg_bound=2):
    """Yield (label, candidates) by elementary S-row operations."""
    p = len(base_gens)
    monos = []
    for k in range(deg_bound + 1):
        for tup in WeightedIntegerVectors(k, [1]*A.n):
            monos.append(prod(A.v[i]**int(tup[i]) for i in range(A.n)))
    for _ in range(n_trials):
        i = random.randrange(p)
        j = random.randrange(p)
        while j == i:
            j = random.randrange(p)
        h = random.choice(monos)
        cand = list(base_gens)
        cand[i] = base_gens[i] + h * base_gens[j]
        yield (f"G[{i}] += ({h})*G[{j}]", cand)

def degenerate_perturbations(A, base_gens):
    """Candidates that are NOT generating sets."""
    p = len(base_gens)
    yield ("two copies of G[1]", [base_gens[1] if i == 0 else base_gens[i] for i in range(p)])
    yield ("x0*G[0] replacing G[0]", [A.v[0]*base_gens[0]] + list(base_gens[1:]))
    drop = [base_gens[i] for i in range(p) if i != 0] + [A.v[0]*base_gens[1]]
    yield ("drop G[0], replace with x0*G[1]", drop)

from sage.combinat.integer_vector_weighted import WeightedIntegerVectors

arrs = [
    HyperplaneArrangement([[1,0,0],[0,1,0],[0,0,1],[1,1,1],[1,1,2]]),
    HyperplaneArrangement(coordinate_vectors(4) + [[1,1,1,0]]),
]

found = False
for A in arrs:
    G = list(A.minimal_generators().gens)
    if len(G) != A.n + 1:
        continue
    print(f"--- Dim {A.n} arrangement ({A.num_planes} planes) ---")
    n_apply, n_gen, n_apply_and_gen = 0, 0, 0
    for label, cand in elementary_perturbations(A, G, n_trials=15, deg_bound=1):
        res = A.check_generalized_saito(generators=cand, verify=True, verbose=False)
        if res['criterion_applies']:
            n_apply += 1
        if res['actually_generates']:
            n_gen += 1
        if res['criterion_applies'] and res['actually_generates']:
            n_apply_and_gen += 1
        if res['counterexample']:
            print(f"  !!! COUNTEREXAMPLE: {label}: ht={res['height']} but does not generate")
            found = True
    print(f"  elementary perturbations: criterion={n_apply}/15, generates={n_gen}/15, both={n_apply_and_gen}/15")
    print(f"  degenerate candidates (criterion should NOT apply):")
    for label, cand in degenerate_perturbations(A, G):
        res = A.check_generalized_saito(generators=cand, verify=True, verbose=False)
        print(f"    {label}: ht={res['height']}, crit={res['criterion_applies']}, gens={res['actually_generates']}")
        if res['counterexample']:
            found = True
            print(f"      !!! COUNTEREXAMPLE")

if not found:
    print("\nNo counterexamples found. Theorem 1.1 holds on all tested cases.")

--- Dim 3 arrangement (5 planes) ---
  elementary perturbations: criterion=15/15, generates=15/15, both=15/15
  degenerate candidates (criterion should NOT apply):
    two copies of G[1]: ht=0, crit=False, gens=False
    x0*G[0] replacing G[0]: ht=1, crit=False, gens=False
    drop G[0], replace with x0*G[1]: ht=0, crit=False, gens=False
--- Dim 4 arrangement (5 planes) ---


  elementary perturbations: criterion=15/15, generates=15/15, both=15/15
  degenerate candidates (criterion should NOT apply):


    two copies of G[1]: ht=0, crit=False, gens=False
    x0*G[0] replacing G[0]: ht=1, crit=False, gens=False


    drop G[0], replace with x0*G[1]: ht=0, crit=False, gens=False

No counterexamples found. Theorem 1.1 holds on all tested cases.


### Low-Height Candidates

If $\operatorname{ht} I_\mathcal{A}(G)<3$, the following candidates are deliberately dependent; the criterion should not apply and they should not generate.


In [23]:
A = HyperplaneArrangement([[1,0,0],[0,1,0],[0,0,1],[1,1,1],[1,1,2]])
G = list(A.minimal_generators().gens)
x = A.v

# Replace G[0] (the Euler vector field) by G[1] -- two generators coincide
Gbad = [G[1], G[1], G[2], G[3]]
res = A.check_generalized_saito(generators=Gbad, verbose=True)
print()

# Replace G[0] by x[1]*G[1] -- still in D(A), but lies in the submodule of G[1..3]
Gbad2 = [x[1]*G[1], G[1], G[2], G[3]]
print()
res2 = A.check_generalized_saito(generators=Gbad2, verbose=True)

  scaled minors (g_i):
    g_0 = 0  (deg -1)
    g_1 = 0  (deg -1)
    g_2 = 0  (deg -1)
    g_3 = 0  (deg -1)
  proper:                     True
  ht I_A(G):                  0
  criterion applies (ht>=3):  False
  all g_i of positive degree: False
  some g_i is linear:         False
  predicts minimal SPOG:      False
  actually generates D(A):    False


  scaled minors (g_i):
    g_0 = 0  (deg -1)
    g_1 = 0  (deg -1)
    g_2 = 0  (deg -1)
    g_3 = 0  (deg -1)
  proper:                     True
  ht I_A(G):                  0
  criterion applies (ht>=3):  False
  all g_i of positive degree: False
  some g_i is linear:         False
  predicts minimal SPOG:      False
  actually generates D(A):    False


### Stress-Test Summary

The tests above separate the three cases needed for using the theorem:

| candidate | height criterion | actual generation | conclusion |
|---|---:|---:|---|
| elementary perturbation of generators | yes | yes | theorem applies |
| repeated generator | no | no | theorem silent |
| dependent candidate | no | no | theorem silent |

No tested candidate satisfies the criterion while failing to generate.


## Example 9: $\ell+1$ generators but Not Strictly Plus-One

**What this shows.** The height criterion can prove generation even when no scaled cofactor is linear.  Such an arrangement is $\ell+1$-generated but not strictly $\ell+1$-generated (SPOG); the extra "one linear coefficient" hypothesis is exactly what fails.


In [24]:
# POG (PD=1) but not SPOG: unique relation has no linear coefficient
A_pog = HyperplaneArrangement([
    [1,0,0],[0,1,0],[0,0,1],[1,1,1],
    [3,1,-2],[3,1,0],[-4,-4,-3],
])
print(f"degrees = {A_pog.degrees()},  is_spog = {A_pog.is_spog()},  num_gens = {len(A_pog.degrees())}")
print(f"free resolution:")
show(A_pog.free_resolution())
print()
res = A_pog.check_generalized_saito(verbose=True)
print()
print(f"Criterion applies (G generates D(A)):  {res['criterion_applies']}")
print(f"But predicts minimal SPOG:             {res['predicts_minimal_spog']}")
print(f"Reason: g_0 = 0 (Euler is a free summand) and the three nonzero g_i are all of degree 2.")

degs = [1, 4, 4, 4],  is_spog = False,  num_gens = 4
free resolution:


<IPython.core.display.Math object>


  scaled minors (g_i):
    g_0 = 0  (deg -1)
    g_1 = -403368*x0^2 - 38416*x0*x1 + 230496*x1^2 + 1591863*x0*x2 + 1591863*x1*x2 + 410571*x2^2  (deg 2)
    g_2 = 38416*x0*x1 + 32928*x1^2 + 119364*x0*x2 + 119364*x1*x2 + 37044*x2^2  (deg 2)
    g_3 = -8232*x1^2 - 29841*x0*x2 - 29841*x1*x2 - 9261*x2^2  (deg 2)
  proper:                     True
  ht I_A(G):                  3
  criterion applies (ht>=3):  True
  all g_i of positive degree: True
  some g_i is linear:         False
  predicts minimal SPOG:      False
  actually generates D(A):    True

Criterion applies (G generates D(A)):  True
But predicts minimal SPOG:             False
Reason: g_0 = 0 (Euler is a free summand) and the three nonzero g_i are all of degree 2.


## Example 10: $\operatorname{pd}D(\mathcal{A})=1$ but rank 2 syzygies

In this case, no $(\ell+1)$-tuple should satisfy the height hypothesis.  We check every four-generator subset in a five-generator example.


In [15]:
import itertools

A_pd1 = HyperplaneArrangement([[1,-2,0],[2,1,1],[0,1,0],[2,-1,2],[-1,0,-1],[-2,2,0],[2,2,-1]])
print(f"degrees = {A_pd1.degrees()},  is_spog = {A_pd1.is_spog()},  num_gens = {len(A_pd1.degrees())}")
print(f"free resolution:")
show(A_pd1.free_resolution())
print()

G = list(A_pd1.minimal_generators().gens)
print(f"l+1 = {A_pd1.n + 1}, |G| = {len(G)}; testing all C(|G|, l+1) = {len(G)} subsets")
print()
any_applies = False
for J in itertools.combinations(range(len(G)), A_pd1.n + 1):
    sub = [G[j] for j in J]
    res = A_pd1.check_generalized_saito(generators=sub, verify=False, verbose=False)
    print(f"  subset {J}: proper={res['is_proper']}, ht={res['height']}, "
          f"criterion_applies={res['criterion_applies']}")
    any_applies = any_applies or res['criterion_applies']


degs = [1, 4, 4, 5, 5],  is_spog = False,  num_gens = 5
free resolution:


<IPython.core.display.Math object>


l+1 = 4, |G| = 5; testing all C(|G|, l+1) = 5 subsets

  subset (0, 1, 2, 3): proper=True, ht=2, criterion_applies=False
  subset (0, 1, 2, 4): proper=True, ht=2, criterion_applies=False
  subset (0, 1, 3, 4): proper=True, ht=2, criterion_applies=False
  subset (0, 2, 3, 4): proper=True, ht=2, criterion_applies=False
  subset (1, 2, 3, 4): proper=True, ht=2, criterion_applies=False


## Example 11: Conjectural Height Bound $\operatorname{ht} I_\mathcal{A}(G)\ge\operatorname{pd}_S D(\mathcal{A})+2$

**Conjecture.** Let $G$ be a homogeneous generating set of $D(\mathcal{A})$.  If $I_\mathcal{A}(G)$ is
proper, then the codimension of the nonfree locus detected by $I_\mathcal{A}(G)$ satisfies the sharp lower
bound
$$
\operatorname{ht} I_\mathcal{A}(G)\ge \operatorname{pd}_S D(\mathcal{A})+2
$$
for the natural classes of arrangements in which the expected determinantal complex is exact.

This unifies the previously tested instances:

* $\operatorname{pd}=0$ (free): bound is vacuous because $I_\mathcal{A}(G)=S$ is not proper.
* $\operatorname{pd}=1$ (POG, including SPOG): bound becomes $\operatorname{ht} I_\mathcal{A}(G)\ge 3$, the hypothesis of the generalized Saito criterion used in Examples 2, 4, 8, 9.
* $\operatorname{pd}\ge 2$: tested below.  By $\operatorname{pd}D(\mathcal{A})\le \ell-2$, so a $\operatorname{pd}=2$ test requires $\ell\ge 4$ and a $\operatorname{pd}=3$ test requires $\ell\ge 5$.


In [11]:
def pd_DA(A):
    """Projective dimension pd_S D(A) read from the minimal free resolution."""
    return A.free_resolution()._length - 1

def height_bound_report(A, label=""):
    """Print ht I_A(G), pd D(A), and whether the conjecture ht>=pd+2 holds."""
    G = list(A.minimal_generators())
    I = A.scaled_minor_ideal(G)
    is_proper = not I.is_one()
    ht = A.scaled_minor_ideal_height(G)
    pd = pd_DA(A)
    bound = pd + 2
    if not is_proper:
        verdict = "vacuous (I_A(G)=S)"
        ok = True
    else:
        ok = ht >= bound
        verdict = ("OK ht>=pd+2" if ok else "FAILS ht<pd+2")
    print(f"  {label:32s}  ell={A.n}  |A|={A.num_planes:2d}  |G|={len(G):2d}  degrees={A.degrees()}")
    print(f"    pd D(A)={pd}   ht I_A(G)={ht}   pd+2={bound}   ->  {verdict}")
    return {"pd": pd, "ht": ht, "is_proper": is_proper, "ok": ok}

print("=== Baselines: pd = 0 (free) and pd = 1 (POG/SPOG) ===\n")

# pd = 0 baselines: I_A(G) is improper, so the conjecture is vacuously satisfied.
height_bound_report(HyperplaneArrangement([[1,0,0],[0,1,0],[0,0,1]]),
                    label="free dim 3 (Boolean)")
height_bound_report(HyperplaneArrangement([[1,0,0],[0,1,0],[0,0,1],[1,-1,0],[1,0,-1],[0,1,-1]]),
                    label="free dim 3 (A_2)")

# pd = 1 baselines: conjecture becomes ht>=3, matching the generalized Saito criterion.
height_bound_report(HyperplaneArrangement([[1,0,0],[0,1,0],[0,0,1],[1,1,1],[1,1,2]]),
                    label="SPOG dim 3")
height_bound_report(HyperplaneArrangement(coordinate_vectors(4)+[[1,1,1,0]]),
                    label="SPOG dim 4")
height_bound_report(HyperplaneArrangement([[1,0,0],[0,1,0],[0,0,1],[1,1,1],
                                   [3,1,-2],[3,1,0],[-4,-4,-3]]),
                    label="POG-not-SPOG dim 3")

# The Example 10 arrangement (Auslander-Buchsbaum forces pd<=1 in ell=3, so this is actually POG
# with two relations rather than pd=2).  Included as a sanity check.
height_bound_report(
    HyperplaneArrangement([[1,-2,0],[2,1,1],[0,1,0],[2,-1,2],[-1,0,-1],[-2,2,0],[2,2,-1]]),
    label="Ex10 (ell=3, pd in fact 1)",
)


=== Baselines: pd = 0 (free) and pd = 1 (POG/SPOG) ===

  free dim 3 (Boolean)              ell=3  |A|= 3  |G|= 3  degs=[1, 1, 1]
    pd D(A)=0   ht I_A(G)=inf   pd+2=2   ->  vacuous (I_A(G)=S)
  free dim 3 (A_2)                  ell=3  |A|= 6  |G|= 3  degs=[1, 2, 3]
    pd D(A)=0   ht I_A(G)=inf   pd+2=2   ->  vacuous (I_A(G)=S)
  SPOG dim 3                        ell=3  |A|= 5  |G|= 4  degs=[1, 2, 3, 3]
    pd D(A)=1   ht I_A(G)=3   pd+2=3   ->  OK ht>=pd+2
  SPOG dim 4                        ell=4  |A|= 5  |G|= 5  degs=[1, 1, 2, 2, 2]
    pd D(A)=1   ht I_A(G)=3   pd+2=3   ->  OK ht>=pd+2
  POG-not-SPOG dim 3                ell=3  |A|= 7  |G|= 4  degs=[1, 4, 4, 4]
    pd D(A)=1   ht I_A(G)=3   pd+2=3   ->  OK ht>=pd+2
  Ex10 (ell=3, pd in fact 1)        ell=3  |A|= 7  |G|= 5  degs=[1, 4, 4, 5, 5]
    pd D(A)=1   ht I_A(G)=3   pd+2=3   ->  OK ht>=pd+2


{'pd': 1, 'ht': 3, 'is_proper': True, 'ok': True}

### Multiple Cases with $\operatorname{pd}D(\mathcal{A})=2$ in Dimension $\ell=4$

For a generic central arrangement in $\mathbb{K}^4$ with $|\mathcal{A}|\ge 2\ell-1=7$ hyperplanes one expects
$\operatorname{pd}_S D(\mathcal{A})=\ell-2=2$.  In that case the conjecture predicts
$\operatorname{ht} I_\mathcal{A}(G)\ge 4$, which by $\operatorname{ht}\le \ell=4$ forces equality.


In [ ]:
print("=== pd D(A) = 2 (dim 4 generic) ===\n")

# Several seeds: the conjecture predicts ht = pd + 2 = 4 = ell in each case.
for seed in [0, 1, 2, 11, 17]:
    set_random_seed(int(seed))
    mat = create_generic_arrangement(4, 7)
    height_bound_report(HyperplaneArrangement(mat),
                        label=f"generic dim 4, |A|=7 (seed {seed})")

# Larger generic arrangement in dim 4: pd remains at 2 (the upper bound ell-2),
# while |G| and degree complexity grow.
for seed, k in [(3, 8), (5, 9)]:
    set_random_seed(int(seed))
    mat = create_generic_arrangement(4, k)
    height_bound_report(HyperplaneArrangement(mat),
                        label=f"generic dim 4, |A|={k} (seed {seed})")


### A Non-Generic $\operatorname{pd}=2$ Example

Generic arrangements are not the only source of $\operatorname{pd}=2$ modules in dimension $4$.  Two-hyperplane deletions of the dimension-$4$ arrangement used in Example 7 also have $\operatorname{pd}=2$, and we verify the conjectured bound on them.


In [13]:
mat_a10 = [
    [1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0],
    [-1, 1, 0, 0], [-1, 0, 1, 0], [0, -1, 1, 0],
    [0, 0, 0, 1], [0, 1, -1, 1], [-1, 1, -1, 1], [0, 0, -1, 1]
]
A10 = HyperplaneArrangement(mat_a10)
print("=== Non-generic pd=2 examples (two-hyperplane deletions of the Example 7 arrangement) ===\n")
for J in [[2, 9], [3, 5], [0, 7], [1, 8]]:
    B = A10.deletion(J)
    height_bound_report(B, label=f"A10.deletion({J})")


=== Non-generic pd=2 examples (two-hyperplane deletions of the Example 7 arrangement) ===

  A10.deletion([2, 9])              ell=4  |A|= 8  |G|= 7  degs=[1, 3, 3, 3, 3, 3, 3]
    pd D(A)=2   ht I_A(G)=4   pd+2=4   ->  OK ht>=pd+2
  A10.deletion([3, 5])              ell=4  |A|= 8  |G|= 7  degs=[1, 3, 3, 3, 3, 3, 3]
    pd D(A)=2   ht I_A(G)=4   pd+2=4   ->  OK ht>=pd+2
  A10.deletion([0, 7])              ell=4  |A|= 8  |G|= 7  degs=[1, 3, 3, 3, 3, 3, 3]
    pd D(A)=2   ht I_A(G)=4   pd+2=4   ->  OK ht>=pd+2
  A10.deletion([1, 8])              ell=4  |A|= 8  |G|= 5  degs=[1, 2, 3, 3, 3]
    pd D(A)=1   ht I_A(G)=3   pd+2=3   ->  OK ht>=pd+2


### A $\operatorname{pd}=3$ Case in Dimension $5$

For a generic central arrangement in $\mathbb{K}^5$ with $|\mathcal{A}|\ge 2\ell-1=9$ hyperplanes one expects
$\operatorname{pd}_S D(\mathcal{A})=\ell-2=3$, hence $\operatorname{ht} I_\mathcal{A}(G)\ge 5=\ell$.  This computation can
be slow; the cell below is wrapped in a guard that prints the input but allows the reader to skip it.


In [ ]:
RUN_PD3 = False   # set True to run; the dim-5 minimal-generator and ideal computations are slow

if RUN_PD3:
    set_random_seed(int(21))
    mat = create_generic_arrangement(5, 9)
    print("=== pd D(A) = 3 (dim 5 generic) ===")
    height_bound_report(HyperplaneArrangement(mat), label="generic dim 5, |A|=9")
else:
    print("Skipped (set RUN_PD3 = True to run).")
